# Supporting clinical documentation and image metadata for a case report of a wandering calcified pelvic mass with suspected ovarian auto-amputation Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This dataset describes an anonymised single-patient clinical case report package, including metadata tables and selected manuscript image metadata. In this notebook, all dataset entities are referenced by their `@id` values whenever applicable.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the Croissant schema URL in a variable so it can be reused consistently throughout the notebook.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/frontiers.qyey-4apm/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))

## 2. Metadata Overview
Review core metadata, available record sets, fields, and their IDs.

The dataset package provided for this resource indicates that the `recordSet` list is empty. The cells below verify that directly from the loaded Croissant object and then provide a compact metadata summary for exploration.

In [ ]:
# Convert metadata to a JSON-like representation for inspection if supported by the library
try:
    metadata_json = metadata.to_json()
except Exception:
    metadata_json = None

if metadata_json is not None:
    print('Top-level metadata keys:')
    print(sorted(metadata_json.keys()))
else:
    print('metadata.to_json() is not available; continuing with attribute-based inspection.')

In [ ]:
# Inspect record set definitions using @id references when available
record_sets = getattr(metadata, 'record_set', None)
if record_sets is None:
    record_sets = getattr(metadata, 'recordSet', [])

record_set_summaries = []
for rs in (record_sets or []):
    rs_id = getattr(rs, 'id', None) or getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    fields = getattr(rs, 'field', None) or getattr(rs, 'fields', None) or []
    field_ids = [getattr(f, 'id', None) or getattr(f, '@id', None) for f in fields]
    record_set_summaries.append({
        'record_set_@id': rs_id,
        'record_set_name': rs_name,
        'field_count': len(fields),
        'field_@ids': field_ids,
    })

record_sets_df = pd.DataFrame(record_set_summaries)
record_sets_df

If one or more record sets were present, the usual next step would be to iterate over them and inspect records with code shaped like:

- `dataset.records(record_set='<record_set_@id>')`

Because this dataset appears to define no record sets, we instead explore the dataset metadata itself as the primary analyzable object.

In [ ]:
# Show selected metadata attributes in a tidy table
metadata_summary = {
    'dataset_@id': getattr(metadata, 'id', None),
    'name': getattr(metadata, 'name', None),
    'description': getattr(metadata, 'description', None),
    'identifier': getattr(metadata, 'identifier', None),
    'version': getattr(metadata, 'version', None),
    'datePublished': getattr(metadata, 'date_published', None) or getattr(metadata, 'datePublished', None),
    'license': getattr(metadata, 'license', None),
    'temporalCoverage': getattr(metadata, 'temporal_coverage', None) or getattr(metadata, 'temporalCoverage', None),
    'spatialCoverage': getattr(metadata, 'spatial_coverage', None) or getattr(metadata, 'spatialCoverage', None),
}

pd.DataFrame(metadata_summary.items(), columns=['property', 'value'])

## 3. Data Extraction
Load available analyzable content into pandas DataFrames.

Since the dataset does not expose tabular `RecordSet` entries, we will extract selected metadata lists into separate DataFrames. This preserves the spirit of Croissant-based exploration while remaining faithful to the actual dataset structure.

In [ ]:
def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

dataframes = {}

# Metadata-derived tables
keywords = ensure_list(getattr(metadata, 'keywords', None))
data_use_cases = ensure_list(getattr(metadata, 'data_use_cases', None) or getattr(metadata, 'dataUseCases', None))
data_biases = ensure_list(getattr(metadata, 'data_biases', None) or getattr(metadata, 'dataBiases', None))
data_limitations = ensure_list(getattr(metadata, 'data_limitations', None) or getattr(metadata, 'dataLimitations', None))
personal_sensitive_information = ensure_list(getattr(metadata, 'personal_sensitive_information', None) or getattr(metadata, 'personalSensitiveInformation', None))
data_collection_type = ensure_list(getattr(metadata, 'data_collection_type', None) or getattr(metadata, 'dataCollectionType', None))
data_collection_timeframe = ensure_list(getattr(metadata, 'data_collection_timeframe', None) or getattr(metadata, 'dataCollectionTimeframe', None))

dataframes['keywords'] = pd.DataFrame({'keyword': keywords})
dataframes['dataUseCases'] = pd.DataFrame({'dataUseCase': data_use_cases})
dataframes['dataBiases'] = pd.DataFrame({'dataBias': data_biases})
dataframes['dataLimitations'] = pd.DataFrame({'dataLimitation': data_limitations})
dataframes['personalSensitiveInformation'] = pd.DataFrame({'personalSensitiveInformation': personal_sensitive_information})
dataframes['dataCollectionType'] = pd.DataFrame({'dataCollectionType': data_collection_type})
dataframes['dataCollectionTimeframe'] = pd.DataFrame({'year_or_period': data_collection_timeframe})

for df_name, df in dataframes.items():
    print(f'[{df_name}] shape = {df.shape}')
    display(df.head())

In [ ]:
# Attempt to extract distributions by their @id values if present in metadata
distribution_items = ensure_list(getattr(metadata, 'distribution', None))
distribution_rows = []

for item in distribution_items:
    distribution_rows.append({
        'distribution_@id': getattr(item, 'id', None) or getattr(item, '@id', None) if not isinstance(item, dict) else item.get('@id'),
        'name': getattr(item, 'name', None) if not isinstance(item, dict) else item.get('name'),
        'contentUrl': getattr(item, 'content_url', None) or getattr(item, 'contentUrl', None) if not isinstance(item, dict) else item.get('contentUrl'),
        'encodingFormat': getattr(item, 'encoding_format', None) or getattr(item, 'encodingFormat', None) if not isinstance(item, dict) else item.get('encodingFormat'),
    })

dataframes['distribution'] = pd.DataFrame(distribution_rows)
dataframes['distribution']

## 4. Exploratory Data Analysis (EDA)
Apply simple processing steps to the extracted metadata tables.

Because this dataset is metadata-centric rather than record-centric, the EDA focuses on:

- text length analysis,
- category frequency summaries,
- filtered views of relevant items,
- normalization of derived numeric features.

In [ ]:
# Example numeric analysis on keyword length
record_set_id = 'keywords'
numeric_field = 'keyword_length'

keywords_df = dataframes[record_set_id].copy()
keywords_df[numeric_field] = keywords_df['keyword'].astype(str).str.len()

threshold = 12
filtered_df = keywords_df[keywords_df[numeric_field] > threshold].copy()
print(f'Filtered records with {numeric_field} > {threshold}:')
display(filtered_df.head())

std_value = filtered_df[numeric_field].std()
if pd.notna(std_value) and std_value != 0:
    filtered_df[f'{numeric_field}_normalized'] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
    )
else:
    filtered_df[f'{numeric_field}_normalized'] = 0.0

print(f'Normalized {numeric_field} for filtered records:')
display(filtered_df[[ 'keyword', numeric_field, f'{numeric_field}_normalized' ]].head())

group_field = 'starts_with_letter'
filtered_df[group_field] = filtered_df['keyword'].astype(str).str[0].str.upper()
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f'Grouped data by {group_field}:')
    display(grouped_df)

In [ ]:
# Frequency analysis for selected categorical metadata tables
for table_name, column_name in [
    ('dataUseCases', 'dataUseCase'),
    ('dataBiases', 'dataBias'),
    ('dataLimitations', 'dataLimitation'),
    ('personalSensitiveInformation', 'personalSensitiveInformation'),
    ('dataCollectionType', 'dataCollectionType')
]:
    df = dataframes[table_name].copy()
    if not df.empty and column_name in df.columns:
        freq_df = df[column_name].value_counts(dropna=False).reset_index()
        freq_df.columns = [column_name, 'count']
        print(f'Frequency table for {table_name}:')
        display(freq_df)

In [ ]:
# Derive simple text metrics for longer free-text metadata fields
text_fields = {
    'dataCollection': getattr(metadata, 'data_collection', None) or getattr(metadata, 'dataCollection', None),
    'dataCollectionRawData': getattr(metadata, 'data_collection_raw_data', None) or getattr(metadata, 'dataCollectionRawData', None),
    'dataManipulationProtocol': getattr(metadata, 'data_manipulation_protocol', None) or getattr(metadata, 'dataManipulationProtocol', None),
    'dataSocialImpact': getattr(metadata, 'data_social_impact', None) or getattr(metadata, 'dataSocialImpact', None),
    'description': getattr(metadata, 'description', None),
}

text_metrics = []
for field_id, text_value in text_fields.items():
    text_str = '' if text_value is None else str(text_value)
    text_metrics.append({
        'field_@id_or_name': field_id,
        'character_count': len(text_str),
        'word_count': len(text_str.split()),
    })

text_metrics_df = pd.DataFrame(text_metrics).sort_values('character_count', ascending=False)
dataframes['text_metrics'] = text_metrics_df
text_metrics_df

## 5. Visualization
Visualize distributions and category frequencies derived from the dataset metadata.

In [ ]:
# Visualization 1: keyword length distribution
plt.figure(figsize=(8, 4))
sns.histplot(keywords_df['keyword_length'], bins=min(10, max(1, keywords_df.shape[0])), kde=True)
plt.title('Distribution of Keyword Lengths')
plt.xlabel('Keyword length (characters)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: personal sensitive information categories
psi_df = dataframes['personalSensitiveInformation'].copy()
if not psi_df.empty:
    plt.figure(figsize=(8, 4))
    order = psi_df['personalSensitiveInformation'].value_counts().index
    sns.countplot(data=psi_df, y='personalSensitiveInformation', order=order)
    plt.title('Personal Sensitive Information Categories')
    plt.xlabel('Count')
    plt.ylabel('Category')
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualization 3: text field word counts
plt.figure(figsize=(9, 4))
sns.barplot(data=text_metrics_df, x='word_count', y='field_@id_or_name', palette='viridis')
plt.title('Word Counts for Selected Free-Text Metadata Fields')
plt.xlabel('Word count')
plt.ylabel('Field')
plt.tight_layout()
plt.show()

## 6. Conclusion
This exploration shows that the Croissant resource is primarily metadata-oriented and does not currently expose tabular `RecordSet` entries for direct row-level loading.

Key observations:

- the dataset describes a single-patient clinical case report package,
- important analyzable content is available through metadata lists such as keywords, use cases, biases, limitations, and sensitive information categories,
- the resource includes distribution entries identifiable by `@id`,
- metadata-derived tables can still support lightweight exploratory analysis and visualization.

If a future version of the dataset exposes `RecordSet` objects, the same notebook structure can be extended to load records directly with `dataset.records(record_set='<record_set_@id>')` and analyze field values by their field `@id`s.